In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("OPENAI_API_KEY")


In [ ]:
# =============================================================================
# UAV-LLM DATASET GENERATOR — KAGGLE NOTEBOOK VERSION
# =============================================================================
# 
# HOW TO USE ON KAGGLE:
# 
# 1. Go to https://www.kaggle.com and sign in
# 2. Click "Create" (top left) > "New Notebook"
# 3. In the notebook settings (right sidebar):
#    - Accelerator: None (CPU is fine, we only need internet)
#    - Internet: ON (CRITICAL — toggle this on)
#    - Persistence: Files (so output survives session restart)
# 4. Click "Add-ons" > "Secrets" in the left sidebar
#    - Add a secret named: OPENAI_API_KEY
#    - Paste your OpenAI API key as the value
# 5. Copy-paste this ENTIRE file into one notebook cell
# 6. Run the cell. It will generate all 18,000 samples.
#
# ESTIMATED TIME: 40-70 minutes (with 10 parallel workers)
# ESTIMATED COST: ~$1.30 with gpt-4o-mini
#
# WHY KAGGLE IS FASTER:
# - Kaggle has enterprise-grade network (low latency to OpenAI servers)
# - We use concurrent.futures for 10 parallel API calls
# - No WiFi drops, no laptop sleep, stable 12-hour runtime
# =============================================================================

# --- CELL 1: Setup and Install ---

import subprocess
subprocess.run(["pip", "install", "openai", "-q"])

import json
import os
import time
import random
import hashlib
import concurrent.futures
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

# --- Load API key from Kaggle Secrets ---
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    OPENAI_API_KEY = secrets.get_secret("OPENAI_API_KEY")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("API key loaded from Kaggle Secrets.")
except Exception:
    # Fallback: paste your key here directly (less secure but works)
    os.environ["OPENAI_API_KEY"] = "sk-paste-your-key-here"
    print("WARNING: Using hardcoded API key. Use Kaggle Secrets instead.")

from openai import OpenAI
client = OpenAI()

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "/kaggle/working/dataset_batches"
FINAL_DIR = "/kaggle/working/final_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

MODEL = "gpt-4o-mini"
BATCH_SIZE = 25
TEMPERATURE = 0.9
MAX_TOKENS = 8000
MAX_WORKERS = 10      # parallel API calls (key speedup)
MAX_RETRIES = 3
RETRY_DELAY = 5

# =============================================================================
# SYSTEM PROMPT
# =============================================================================

SYSTEM_PROMPT = """You are a synthetic dataset generator for a research project on privacy-preserving federated learning for LLM-driven UAV (drone) systems. Your job is to generate high-quality pairs of (natural language command, structured JSON output) that simulate how a human drone operator would issue commands and how those commands would be parsed into machine-executable format.

STRICT RULES YOU MUST FOLLOW:

1. OUTPUT FORMAT: Return ONLY a valid JSON array. No markdown, no explanation, no commentary, no code fences, no backticks. Just the raw JSON array starting with [ and ending with ].

2. NATURAL LANGUAGE VARIATION: Every input sentence must be unique. Never repeat phrasing. Vary vocabulary, sentence structure, and formality level across samples. Mix these styles randomly:
   - Formal: "Navigate to coordinates 34.05N, 118.24W at an altitude of 45 meters"
   - Semi-formal: "Go to position 34.05, -118.24, fly at 45m"
   - Casual: "Head over to those coords, keep it around 45 meters high"
   - Terse: "Move to 34.05, -118.24, alt 45"

3. PARAMETER RANGES (never exceed these):
   - latitude: -90.0 to 90.0 (use realistic locations, not random numbers)
   - longitude: -180.0 to 180.0 (use realistic locations)
   - altitude_m: 5 to 400
   - speed_ms: 1.0 to 25.0
   - heading_deg: 0 to 359
   - radius_m: 10 to 2000
   - duration_sec: 10 to 3600
   - distance_m: 5 to 5000
   - zoom_level: 1 to 30
   - interval_sec: 1 to 60

4. JSON OUTPUT SCHEMA: Every sample must follow this structure:
   {
     "input": "natural language command string",
     "output": {
       "command_type": "one of: navigation|surveillance|inspection|emergency|formation|status",
       "action": "specific action string",
       "parameters": { ... relevant parameters only ... },
       "camera_config": { ... only if visual capture is involved, otherwise null ... },
       "constraints": { ... only if safety limits mentioned, otherwise null ... },
       "priority": "one of: critical|high|normal|low",
       "mission_id": "unique string like M-XXXX"
     }
   }

5. VALID ACTIONS PER COMMAND TYPE:
   - navigation: goto_waypoint, follow_path, orbit_point, hover, adjust_altitude, set_speed, return_to_base
   - surveillance: area_scan, perimeter_patrol, track_target, continuous_monitor, capture_image, start_video, stop_video
   - inspection: close_approach, circle_structure, vertical_scan, thermal_scan, zoom_inspect
   - emergency: emergency_land, return_home, avoid_obstacle, abort_mission, signal_lost_protocol
   - formation: maintain_formation, spread_out, converge, follow_leader, swap_position
   - status: report_battery, report_position, report_weather, report_signal, system_diagnostics

6. CAMERA CONFIG OPTIONS (when applicable):
   - mode: photo|video|thermal|multispectral|off
   - interval_sec: 1 to 60 (for photo mode only)
   - resolution: 480p|720p|1080p|4K
   - zoom_level: 1 to 30

7. CONSTRAINTS OPTIONS (when applicable):
   - geofence_radius_m: float
   - min_altitude_m: integer
   - max_altitude_m: integer
   - max_speed_ms: float
   - no_fly_zones: list of zone identifier strings
   - time_limit_sec: integer

8. REALISM: Use real-world plausible coordinates for the specified domain.
9. MISSION IDs: Generate unique IDs in format M-XXXX where XXXX is a random 4-digit number."""

# =============================================================================
# ALL GENERATION PROMPTS
# =============================================================================

PROMPTS = {
    "nav_agri": {
        "total": 1350,
        "prompt": """Generate exactly {bs} samples of NAVIGATION commands for agricultural drone operations.
Context: Drones over farmland, vineyards, orchards. Operators are farmers, agronomists.
- command_type: "navigation" for all. Mix actions: goto_waypoint, follow_path, orbit_point, hover, adjust_altitude, set_speed, return_to_base
- Coordinates in agricultural regions (US Midwest, California, India Punjab, Brazil, Australia)
- Altitudes 10-80m, speeds 3-12 m/s
- 40% formal, 30% semi-formal, 20% casual, 10% terse
- camera_config in 30%, constraints in 20%, priority mostly "normal"
- 5/{bs} compound commands. Batch {bn} — ALL NEW unique content.
Return ONLY the JSON array."""
    },
    "nav_urban": {
        "total": 1350,
        "prompt": """Generate exactly {bs} samples of NAVIGATION commands for urban drone operations.
Context: City drones for delivery, traffic monitoring, construction oversight. Operators are logistics coordinators, city planners.
- command_type: "navigation" for all. Mix all navigation actions.
- Coordinates in real cities (New York, London, Tokyo, Dubai, Sydney, Singapore, Berlin, Mumbai)
- Altitudes 30-150m, speeds 5-15 m/s
- Mix formality levels. Constraints in 40%, no_fly_zones in 3+ samples
- Some reference landmarks not just coordinates
- 5/{bs} compound commands. Batch {bn} — completely fresh content.
Return ONLY the JSON array."""
    },
    "nav_sar": {
        "total": 1350,
        "prompt": """Generate exactly {bs} samples of NAVIGATION commands for search and rescue drones.
Context: Disaster response, missing persons, maritime emergency, wilderness rescue. Operators are emergency responders.
- command_type: "navigation". Mix all navigation actions.
- Coordinates in disaster-prone areas (coasts, mountains, flood plains, forests)
- Altitudes 15-200m, speeds 2-20 m/s
- Tone: 50% terse, 30% semi-formal, 20% formal (urgent style)
- camera_config in 50%, priority 40% critical, 40% high, 20% normal
- time_limit_sec in 30% of constraints. 6/{bs} compound. Batch {bn} — unique content.
Return ONLY the JSON array."""
    },
    "nav_security": {
        "total": 1350,
        "prompt": """Generate exactly {bs} samples of NAVIGATION commands for border/security surveillance drones.
Context: Border patrol, coastline monitoring, facility perimeter security. Operators are security/military personnel.
- command_type: "navigation". Mix all navigation actions.
- Coordinates in realistic border/coastal regions worldwide
- Altitudes 50-300m, speeds 8-20 m/s. Formal/terse tone.
- Constraints in 50%, no_fly_zones frequently, camera_config in 40%
- Priority mix of high and normal. 5/{bs} compound. Batch {bn} — new scenarios.
Return ONLY the JSON array."""
    },
    "surv_agri": {
        "total": 1125,
        "prompt": """Generate exactly {bs} samples of SURVEILLANCE commands for agricultural monitoring.
Context: Crop health, pest detection, irrigation assessment, livestock tracking. Operators are farm managers.
- command_type: "surveillance". Mix: area_scan, perimeter_patrol, track_target, continuous_monitor, capture_image, start_video, stop_video
- camera_config REQUIRED: photo, video, thermal, multispectral modes
- Agricultural coordinates, altitudes 10-60m
- Include radius_m, duration_sec. Constraints in 25%. Priority mostly normal.
- 6/{bs} compound. Some vague commands like "check the wheat field". Batch {bn}.
Return ONLY the JSON array."""
    },
    "surv_urban": {
        "total": 1125,
        "prompt": """Generate exactly {bs} samples of SURVEILLANCE commands for urban security operations.
Context: Traffic monitoring, crowd surveillance, building perimeter, public safety. Operators are security teams.
- command_type: "surveillance". Mix all surveillance actions.
- camera_config REQUIRED: mostly video/photo, thermal for night ops. 1080p/4K resolution.
- Urban coordinates. Altitudes 40-150m. High zoom_level values.
- Constraints with time_limit_sec and geofence frequently. Priority mix high/normal.
- Reference scenarios: "monitor stadium entrance", "track vehicle on 5th Ave"
- 5/{bs} compound. Batch {bn} — fresh content.
Return ONLY the JSON array."""
    },
    "surv_env": {
        "total": 1125,
        "prompt": """Generate exactly {bs} samples of SURVEILLANCE commands for environmental monitoring.
Context: Forest fire detection, wildlife tracking, flood assessment, water quality. Operators are scientists, rangers.
- command_type: "surveillance". Mix all surveillance actions.
- camera_config REQUIRED: thermal for fire, multispectral for vegetation, video for wildlife
- Natural area coordinates (forests, wetlands, national parks). Altitudes 20-300m.
- Large radius_m, long duration_sec. Priority critical for fire, normal for routine.
- Scientist/ranger speaking style. 5/{bs} compound. Batch {bn}.
Return ONLY the JSON array."""
    },
    "surv_maritime": {
        "total": 1125,
        "prompt": """Generate exactly {bs} samples of SURVEILLANCE commands for maritime/coastal operations.
Context: Shipping lane monitoring, oil spill detection, vessel tracking, port security. Operators are coast guard, marine researchers.
- command_type: "surveillance". Mix all surveillance actions.
- camera_config REQUIRED: thermal for night vessels, multispectral for oil spills
- Coastal/shipping route coordinates. Altitudes 30-250m. Large areas, long durations.
- Wind constraints realistic for maritime. Mix all priority levels.
- 5/{bs} compound. Batch {bn} — unique content.
Return ONLY the JSON array."""
    },
    "insp_infra": {
        "total": 1350,
        "prompt": """Generate exactly {bs} samples of INSPECTION commands for infrastructure.
Context: Bridges, power lines, wind turbines, cell towers, pipelines, dams. Operators are engineers, technicians.
- command_type: "inspection". Mix: close_approach, circle_structure, vertical_scan, thermal_scan, zoom_inspect
- camera_config REQUIRED: 4K photo, thermal for heat anomalies, high zoom (10-30x)
- distance_m: 3-15m. Precise altitudes. Slow speeds 1-4 m/s. angle_deg for many.
- Constraints: min distance from structure. Priority normal/high.
- 5/{bs} compound (approach + inspect). Batch {bn}.
Return ONLY the JSON array."""
    },
    "insp_energy": {
        "total": 1350,
        "prompt": """Generate exactly {bs} samples of INSPECTION commands for energy/industrial sites.
Context: Solar panels, oil rigs, factories, warehouses, mines, construction. Operators are energy engineers, safety inspectors.
- command_type: "inspection". Mix all inspection actions.
- camera_config REQUIRED: thermal for solar/electrical, multispectral for corrosion, 4K photo
- Grid scan patterns, spiral around structures. Altitudes 5-50m. Speeds 1-3 m/s.
- Constraints: restricted airspace. Priority high for safety, normal for routine.
- 5/{bs} compound. Batch {bn} — fresh scenarios.
Return ONLY the JSON array."""
    },
    "emergency": {
        "total": 1800,
        "prompt": """Generate exactly {bs} samples of EMERGENCY commands for drones.
Context: Battery failure, signal loss, obstacle detection, weather, airspace violation, mechanical failure. Operators are panicked/rushed or automated alerts.
- command_type: "emergency". Mix: emergency_land, return_home, avoid_obstacle, abort_mission, signal_lost_protocol
- Minimal parameters: landing coords, safe altitude, obstacle direction
- camera_config mostly null (few with video to record emergency)
- Priority almost all "critical". Tone: terse, urgent, short sentences.
  Good: "Land NOW. Battery critical." / "RTB immediately, storm approaching" / "Abort. Manned aircraft."
- 3/{bs} should be SYSTEM ALERTs. Only 2-3 compound. Include bird strike, GPS jamming, wind gust, geofence breach.
- Batch {bn}.
Return ONLY the JSON array."""
    },
    "formation": {
        "total": 1800,
        "prompt": """Generate exactly {bs} samples of FORMATION commands for multi-drone swarms.
Context: 3-20 drones in coordinated patterns for survey, synchronized surveillance, relay. Operators are swarm coordinators.
- command_type: "formation". Mix: maintain_formation, spread_out, converge, follow_leader, swap_position
- Parameters: formation_type (line/grid/v_shape/circle/diamond), spacing_m (20-200), reference_drone_id, heading_deg, altitude_m, speed_ms
- camera_config in 30%. Geofence constraints. Priority mostly normal.
- Formal/precise tone. 5/{bs} compound. Address specific drones: "Drone 3, swap with Drone 5".
- Batch {bn} — new scenarios.
Return ONLY the JSON array."""
    },
    "status": {
        "total": 1800,
        "prompt": """Generate exactly {bs} samples of STATUS query commands for drones.
Context: Operators requesting drone state, health, environment info. Questions, not actions.
- command_type: "status". Mix: report_battery, report_position, report_weather, report_signal, system_diagnostics
- Minimal parameters: drone_id, metric, units. camera_config: null. constraints: null.
- Priority mostly normal, high for battery-critical.
- Tone: very varied (casual "how much battery left?", formal "report GPS coordinates of UAV-07", terse "battery status all units")
- 3/{bs} compound queries (multiple metrics). Include weather, flight time estimates.
- Batch {bn}.
Return ONLY the JSON array."""
    },
    "ambiguous": {
        "total": 450,
        "prompt": """Generate exactly {bs} samples of AMBIGUOUS/UNDERSPECIFIED commands.
Context: Real operators giving incomplete, vague, or contradictory commands.
- command_type: best guess of intent. action: most likely action.
- Include: "status": "needs_clarification", "missing_fields": [...], "assumed_defaults": {{...}}
- Examples: "Go up" (missing altitude), "Take a picture" (missing location), "Come back" (ambiguous destination), "Faster" (missing target speed), "Check that area" (missing specifics), "Same as yesterday" (no memory)
- Casual/conversational tone. Priority mostly normal. Batch {bn}.
Return ONLY the JSON array."""
    },
    "compound": {
        "total": 450,
        "prompt": """Generate exactly {bs} samples of COMPLEX COMPOUND commands chaining 3-5 actions.
Context: Experienced operators issuing full mission plans in one paragraph.
- output must have "action_sequence" array with 3-5 action objects, each with own action, parameters, camera_config
- command_type: "compound". Input is multi-sentence mission paragraph.
- Mix domains: agricultural, urban, SAR, inspection, maritime.
- Mix formality. Priority varies by urgency. Batch {bn}.
Return ONLY the JSON array."""
    },
}

# =============================================================================
# API CALL WITH RETRY
# =============================================================================

def call_api(prompt_text, batch_num):
    """Single API call with retry logic. Returns (samples_list, token_usage)."""
    formatted = prompt_text.format(bs=BATCH_SIZE, bn=batch_num)
    
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": formatted}
                ]
            )
            
            raw = response.choices[0].message.content.strip()
            tokens = {
                "input": response.usage.prompt_tokens,
                "output": response.usage.completion_tokens
            }
            
            # Clean and parse
            text = raw
            if text.startswith("```json"): text = text[7:]
            elif text.startswith("```"): text = text[3:]
            if text.endswith("```"): text = text[:-3]
            text = text.strip()
            
            start = text.find("[")
            end = text.rfind("]")
            if start == -1 or end == -1:
                raise ValueError("No JSON array found")
            text = text[start:end+1]
            
            # Fix trailing commas
            import re
            text = re.sub(r',\s*}', '}', text)
            text = re.sub(r',\s*]', ']', text)
            
            samples = json.loads(text)
            if not isinstance(samples, list) or len(samples) == 0:
                raise ValueError(f"Invalid result: got {type(samples)}")
            
            return samples, tokens
            
        except Exception as e:
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)
            else:
                raise e

# =============================================================================
# PARALLEL BATCH PROCESSOR
# =============================================================================

def process_single_batch(args):
    """Process one batch. Used by ThreadPoolExecutor."""
    prompt_key, batch_num, prompt_text = args
    batch_id = f"{prompt_key}_b{batch_num}"
    batch_file = os.path.join(OUTPUT_DIR, f"{batch_id}.json")
    
    # Skip if already exists
    if os.path.exists(batch_file):
        try:
            with open(batch_file) as f:
                existing = json.load(f)
            if isinstance(existing, list) and len(existing) > 0:
                return {"id": batch_id, "status": "skipped", "samples": len(existing), "tokens": {"input": 0, "output": 0}}
        except:
            pass  # file is corrupt, regenerate
    
    try:
        samples, tokens = call_api(prompt_text, batch_num)
        
        with open(batch_file, 'w') as f:
            json.dump(samples, f, indent=2)
        
        return {"id": batch_id, "status": "ok", "samples": len(samples), "tokens": tokens}
    
    except Exception as e:
        return {"id": batch_id, "status": "failed", "error": str(e), "samples": 0, "tokens": {"input": 0, "output": 0}}


def generate_all():
    """Main generation loop with parallel execution."""
    
    # Build task list
    tasks = []
    for key, config in PROMPTS.items():
        n_batches = config["total"] // BATCH_SIZE
        for bn in range(1, n_batches + 1):
            tasks.append((key, bn, config["prompt"]))
    
    total = len(tasks)
    print(f"Total batches to process: {total}")
    print(f"Parallel workers: {MAX_WORKERS}")
    print(f"Model: {MODEL}")
    print(f"Batch size: {BATCH_SIZE}")
    print("=" * 60)
    
    # Check how many already exist
    existing = sum(1 for t in tasks if os.path.exists(os.path.join(OUTPUT_DIR, f"{t[0]}_b{t[1]}.json")))
    print(f"Already completed: {existing}")
    print(f"Remaining: {total - existing}")
    print("=" * 60)
    
    completed = 0
    failed = 0
    skipped = 0
    total_samples = 0
    total_input_tokens = 0
    total_output_tokens = 0
    start_time = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_single_batch, task): task for task in tasks}
        
        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            
            if result["status"] == "ok":
                completed += 1
                total_samples += result["samples"]
                total_input_tokens += result["tokens"]["input"]
                total_output_tokens += result["tokens"]["output"]
            elif result["status"] == "skipped":
                skipped += 1
                total_samples += result["samples"]
            else:
                failed += 1
            
            done = completed + skipped + failed
            elapsed = time.time() - start_time
            rate = (completed) / elapsed * 3600 if elapsed > 0 and completed > 0 else 0
            eta_min = (total - done) / (done / elapsed) / 60 if done > 0 and elapsed > 0 else 0
            
            if done % 10 == 0 or result["status"] == "failed":
                print(f"[{done}/{total}] "
                      f"OK:{completed} Skip:{skipped} Fail:{failed} | "
                      f"Samples:{total_samples} | "
                      f"Rate:{rate:.0f} batches/hr | "
                      f"ETA:{eta_min:.0f}min | "
                      f"{'FAIL: ' + result.get('error','')[:50] if result['status']=='failed' else result['id']}")
    
    elapsed = time.time() - start_time
    cost = (total_input_tokens / 1e6 * 0.15) + (total_output_tokens / 1e6 * 0.60)
    
    print(f"\n{'='*60}")
    print(f"GENERATION COMPLETE")
    print(f"{'='*60}")
    print(f"Time: {elapsed/60:.1f} minutes")
    print(f"Completed: {completed} | Skipped: {skipped} | Failed: {failed}")
    print(f"Total samples: {total_samples}")
    print(f"Tokens: {total_input_tokens:,} input + {total_output_tokens:,} output")
    print(f"Estimated cost: ${cost:.2f}")
    
    return total_samples

# =============================================================================
# VALIDATION
# =============================================================================

VALID_ACTIONS = {
    "navigation": ["goto_waypoint", "follow_path", "orbit_point", "hover", "adjust_altitude", "set_speed", "return_to_base"],
    "surveillance": ["area_scan", "perimeter_patrol", "track_target", "continuous_monitor", "capture_image", "start_video", "stop_video"],
    "inspection": ["close_approach", "circle_structure", "vertical_scan", "thermal_scan", "zoom_inspect"],
    "emergency": ["emergency_land", "return_home", "avoid_obstacle", "abort_mission", "signal_lost_protocol"],
    "formation": ["maintain_formation", "spread_out", "converge", "follow_leader", "swap_position"],
    "status": ["report_battery", "report_position", "report_weather", "report_signal", "system_diagnostics"]
}

PARAM_RANGES = {
    "latitude": (-90, 90), "longitude": (-180, 180), "altitude_m": (5, 400),
    "speed_ms": (1.0, 25.0), "heading_deg": (0, 359), "radius_m": (10, 2000),
    "duration_sec": (10, 3600), "distance_m": (5, 5000), "zoom_level": (1, 30),
    "interval_sec": (1, 60)
}

def validate_dataset(data):
    errors = []
    type_counts = Counter()
    action_counts = Counter()
    seen = set()
    dupes = 0
    
    for i, s in enumerate(data):
        inp = s.get("input", "")
        h = hashlib.md5(inp.encode()).hexdigest()
        if h in seen: dupes += 1
        seen.add(h)
        
        out = s.get("output", {})
        ct = out.get("command_type", "unknown")
        type_counts[ct] += 1
        
        if "action" in out:
            action_counts[out["action"]] += 1
            if ct in VALID_ACTIONS and out["action"] not in VALID_ACTIONS[ct]:
                errors.append(f"Sample {i}: Invalid action '{out['action']}' for '{ct}'")
        
        params = out.get("parameters", {})
        if isinstance(params, dict):
            for k, v in params.items():
                if k in PARAM_RANGES and isinstance(v, (int, float)):
                    lo, hi = PARAM_RANGES[k]
                    if v < lo or v > hi:
                        errors.append(f"Sample {i}: {k}={v} out of [{lo},{hi}]")
        
        pri = out.get("priority", "")
        if pri and pri not in ["critical", "high", "normal", "low"]:
            errors.append(f"Sample {i}: Invalid priority '{pri}'")
    
    return errors, {
        "total": len(data), "unique": len(seen), "duplicates": dupes,
        "errors": len(errors), "types": dict(type_counts.most_common()),
        "top_actions": dict(action_counts.most_common(15))
    }

# =============================================================================
# MERGE, DEDUPLICATE, SPLIT
# =============================================================================

def merge_and_split():
    import glob
    import numpy as np
    
    print("STEP 1: Merging batches...")
    all_samples = []
    for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.json"))):
        if "log" in f: continue
        try:
            with open(f) as fh:
                batch = json.load(fh)
            if isinstance(batch, list):
                all_samples.extend(batch)
        except: pass
    print(f"  Merged: {len(all_samples)} samples")
    
    print("\nSTEP 2: Deduplicating...")
    seen = set()
    unique = []
    for s in all_samples:
        h = hashlib.md5(s.get("input","").encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(s)
    print(f"  Removed {len(all_samples)-len(unique)} duplicates. Remaining: {len(unique)}")
    all_samples = unique
    
    print("\nSTEP 3: Validating...")
    errors, stats = validate_dataset(all_samples)
    print(f"  Total: {stats['total']} | Unique: {stats['unique']} | Errors: {stats['errors']}")
    print(f"  Distribution: {stats['types']}")
    if errors:
        print(f"  First 10 errors:")
        for e in errors[:10]: print(f"    {e}")
    
    print("\nSTEP 4: Saving full dataset...")
    full_path = os.path.join(FINAL_DIR, "uav_llm_dataset.json")
    with open(full_path, 'w') as f:
        json.dump(all_samples, f, indent=2)
    print(f"  Saved: {full_path}")
    
    print("\nSTEP 5: Train/Val/Test split...")
    random.seed(42)
    shuffled = all_samples.copy()
    random.shuffle(shuffled)
    n = len(shuffled)
    tr_end = int(n * 0.833)
    va_end = tr_end + int(n * 0.0833)
    train, val, test = shuffled[:tr_end], shuffled[tr_end:va_end], shuffled[va_end:]
    
    for name, split in [("train", train), ("val", val), ("test", test)]:
        p = os.path.join(FINAL_DIR, f"{name}.json")
        with open(p, 'w') as f: json.dump(split, f, indent=2)
        print(f"  {name}: {len(split)} -> {p}")
    
    print("\nSTEP 6: Federated splits...")
    np.random.seed(42)
    
    for n_clients in [5, 10]:
        for alpha in [0.1, 0.5, 1.0, 10.0]:
            # Group by type
            type_idx = defaultdict(list)
            for i, s in enumerate(train):
                type_idx[s["output"]["command_type"]].append(i)
            
            client_idx = [[] for _ in range(n_clients)]
            for ct, indices in type_idx.items():
                np.random.shuffle(indices)
                props = np.random.dirichlet([alpha] * n_clients)
                splits = (props * len(indices)).astype(int)
                splits[-1] = len(indices) - splits[:-1].sum()
                start = 0
                for c, count in enumerate(splits):
                    client_idx[c].extend(indices[start:start+count])
                    start += count
            
            fed_dir = os.path.join(FINAL_DIR, f"fed_n{n_clients}_a{alpha}")
            os.makedirs(fed_dir, exist_ok=True)
            
            summary = []
            for c in range(n_clients):
                np.random.shuffle(client_idx[c])
                cdata = [train[i] for i in client_idx[c]]
                cp = os.path.join(fed_dir, f"client_{c}.json")
                with open(cp, 'w') as f: json.dump(cdata, f, indent=2)
                types = Counter(s["output"]["command_type"] for s in cdata)
                dom = types.most_common(1)[0] if types else ("?", 0)
                summary.append(f"C{c}:{len(cdata)}({dom[0]}={dom[1]})")
            
            print(f"  n={n_clients} a={alpha}: {' | '.join(summary)}")
    
    print("\nSTEP 7: Dataset report...")
    report = {
        "date": datetime.now().isoformat(),
        "total": len(all_samples),
        "train": len(train), "val": len(val), "test": len(test),
        "types": stats["types"],
        "actions": stats["top_actions"],
        "errors": stats["errors"],
    }
    rp = os.path.join(FINAL_DIR, "report.json")
    with open(rp, 'w') as f: json.dump(report, f, indent=2)
    
    print(f"\nDONE. All files in: {FINAL_DIR}/")
    print(f"\nTo download: click the folder icon in Kaggle sidebar,")
    print(f"navigate to /kaggle/working/final_dataset/, right-click > Download")
    
    return stats

# =============================================================================
# RUN EVERYTHING
# =============================================================================

print("=" * 60)
print("UAV-LLM DATASET GENERATOR — KAGGLE EDITION")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")
print()

# Phase 1: Generate
total = generate_all()

print()

# Phase 2: Merge and Split
if total > 0:
    stats = merge_and_split()

print()
print(f"End time: {datetime.now().strftime('%H:%M:%S')}")
print("All done. Download your dataset from /kaggle/working/final_dataset/")

In [ ]:
# =============================================================================
# DOWNLOAD DATASET AS ZIP — Run this in a NEW Kaggle cell after generation
# =============================================================================

import os
import zipfile
from datetime import datetime

# Paths
FINAL_DIR = "/kaggle/working/final_dataset"
ZIP_PATH = "/kaggle/working/uav_llm_dataset.zip"

# Create zip
print("Creating zip file...")
file_count = 0

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(FINAL_DIR):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, "/kaggle/working")
            zf.write(filepath, arcname)
            file_count += 1
            print(f"  Added: {arcname}")

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"\nDone. {file_count} files packed.")
print(f"Zip size: {size_mb:.1f} MB")
print(f"Location: {ZIP_PATH}")

# Auto-download in Kaggle
try:
    from IPython.display import FileLink
    display(FileLink('uav_llm_dataset.zip'))
    print("\nClick the link above to download.")
except:
    print("\nTo download: Kaggle sidebar → folder icon → uav_llm_dataset.zip → Download")

In [ ]:
import os
# Delete the specific file
os.remove("/kaggle/working/final_dataset/uav_llm_dataset.jsonn")
print("Deleted.")